# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [47]:
# TODO: Import the necessary libs
import os
import chromadb

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from chromadb.utils import embedding_functions

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from lib.parsers import PydanticOutputParser

from tavily import TavilyClient # for web search

In [49]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [50]:
# Create Pydantic model for structured LLM outputs
# This ensures reliable routing logic that determines useful or not, and LLM returns parsable
# boolean output
class EvaluationReport(BaseModel):
    """Structured output for retrieval evaluation."""
    is_useful: bool = Field(
        description="Whether the documents are useful to answer the question"
    )
    description: str = Field(
        description="Detailed explanation of the evaluation result"
    )

In [51]:
# It should use chroma client and load collection you created
chroma_client = chromadb.PersistentClient(path="chromadb")

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY
)

# get existing collection
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

In [52]:
# ============================================================================
# LONG-TERM MEMORY SETUP - using chromaDB collection with OpenAI embeddings
# ============================================================================
print("\n" + "="*80)
print("Setting up Long-Term Memory System")
print("="*80)

def set_long_term_memory(openai_api_key):
    """
    Create a separate ChromaDB collection for learned knowledge
    """
    chroma_client = chromadb.PersistentClient(path="chromadb")
    embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
        api_key=openai_api_key
    )

    learned_collection = chroma_client.get_or_create_collection(
        name="udaplay_learned",
        embedding_function=embedding_fn
    )

    return learned_collection

# Initialize long-term memory
learned_collection = set_long_term_memory(OPENAI_API_KEY)
print(f" Long-term memory collection ready")
print(f" Main collection: {collection.count()} documents")
print(f" Learned collection: {learned_collection.count()} documents")
print("="*80 + "\n")


Setting up Long-Term Memory System
 Long-term memory collection ready
 Main collection: 15 documents
 Learned collection: 0 documents



### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [53]:
# TODO: Create retrieve_game tool
# Tool Docstring:
@tool
def retrieve_game(query: str):
    """
    Semantic search: Searches BOTH original game database and learned knowledge

    args:
    - query: a question about game industry. 
    
    You'll receive results from:
    - Original Database containing:
        - Platform: like Game Boy, Playstation 5, Xbox 360...)
        - Name: Name of the Game
        - YearOfRelease: Year when that game was released for that platform
        - Description: Additional details about the game
    - Learned Knowledge containing:
        - Information previously acquired from web searches
    
        Each result includes source type so you can cite approprately
    """

    # Search original collection
    original_results = collection.query(
        query_texts=[query],
        n_results=3, # Retrieve top 3 matches
        include=['documents', 'metadatas', 'distances']
    )

    # Search learned knowledge collection
    try:
        learned_results = learned_collection.query(
            query_texts=[query],
            n_results=2,
            include=['documents', 'metadatas', 'distances']
        )
    except Exception as e:
        # if learned collection has issues, continue with just original
        learned_results = {'documents': [[]], 'metadatas': [[]], 'distances': [[]]}

    # Build formatted results for agent response    
    formatted_results = []

    # Add original database results
    if original_results['documents'][0]:
        formatted_results.append("===ORIGINAL GAME DATABASE ===")

        for i, metadata in enumerate(original_results['metadatas'][0]):
            formatted_results.append(
                f"\nGame {i+1}:\n"
                f"Name: {metadata['Name']}\n"
                f"Platform: {metadata['Platform']}\n"
                f"Year of Release: {metadata['YearOfRelease']}\n"
                f"Genre: {metadata.get('Genre', 'N/A')}\n"
                f"Publisher: {metadata.get('Publisher', 'N/A')}\n"
                f"Description: {metadata['Description']}\n"
            )

    # Add learned knowledge results
    if learned_results['documents'][0]:
        formatted_results.append("\n=== LEARNED KNOWLEDGE (from previous web searches) ===")

        for i, (doc, metadata) in enumerate(zip(learned_results['documents'][0], learned_results['metadatas'][0])):
            formatted_results.append(
                f"\nLearned Info {i+1}:\n"
                f" Title: {metadata.get('title', 'N/A')}\n"
                f" Source URL: {metadata.get('url', 'N/A')}\n"
                f" Learned from query: {metadata.get('query', 'N/A')}\n"
                f" Content: {doc[:300]}...\n"
            )    

    # Format results into a readable string for agent
    if not original_results['documents'][0] and not learned_results['documents'][0]:
        return "No results found in original database or learned knowledge."

    return "\n".join(formatted_results)

In [54]:
# Test the tool directly
result = retrieve_game("Mario games")
print(result) # expected output: formatted info with all fields

===ORIGINAL GAME DATABASE ===

Game 1:
Name: Super Mario 64
Platform: Nintendo 64
Year of Release: 1996
Genre: Platformer
Publisher: Nintendo
Description: A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.


Game 2:
Name: Super Mario World
Platform: Super Nintendo Entertainment System (SNES)
Year of Release: 1990
Genre: Platformer
Publisher: Nintendo
Description: A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.


Game 3:
Name: Mario Kart 8 Deluxe
Platform: Nintendo Switch
Year of Release: 2017
Genre: Racing
Publisher: Nintendo
Description: An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.



#### Evaluate Retrieval Tool

In [55]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."

# Use EvaluationReport to parse the result
# Tool Docstring:
@tool
def evaluate_retrieval(question: str, retrieve_docs: str):
    """
    Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question. 

    args: 
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """

    # Initialize LLM as judge
    llm_judge = LLM(model="gpt-4o-mini", temperature=0.3)

    evaluation_prompt = f"""Your task is to evaluate if the documents are enough to respond to the query.
                        Give a detailed explanation, so it's possible to take action to accept it or not.

                        Question: {question}
                        Retrieved Documents: {retrieve_docs}

    
                        Evaluate whether the documents contain sufficient information to answer the question.

                        Consider:
                        - Do the documents directly address the question?
                        - Is the information specific and accurate?
                        - Are there any gaps in the information needed?

                        Provide your evaluation as JSON with 'userful' (boolean) and 'description' (string) fields."""
    
    # get structured response from judge LLM
    response = llm_judge.invoke(
        input=evaluation_prompt,
        response_format=EvaluationReport
    )

    # parse with Pydantic parser
    parser = PydanticOutputParser(model_class=EvaluationReport)

    try:
        evaluation = parser.parse(response)
        return f"Userful: {evaluation.is_useful}\nReasoning: {evaluation.description}"
    except Exception as e:
        return f"Userful: False\nReasoning: Failed to parse evaluation response - Error: {str(e)}"


In [56]:
# Test the evaluation tool directly
sample_docs = "Game 1: Super Mario 64 (Nintendo 64, 1996)"
result = evaluate_retrieval(
    question="When was Mario 64 released?",
    retrieve_docs=sample_docs
)

print(result) # expected output: "Useful: True/False\nReasoning: detailed explanation

Userful: True
Reasoning: The retrieved document provides a clear and direct answer to the question regarding the release date of Mario 64. It specifies that 'Super Mario 64' was released for the Nintendo 64 in 1996. This information is both specific and accurate, directly addressing the query without any ambiguity. There are no gaps in the information needed to answer the question, as the release year is explicitly stated. Therefore, the document is sufficient to respond to the question.


#### Game Web Search Tool

In [57]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
@tool
def game_web_search(question: str):
    """
    Search the web for gaming industry information.

    Automatically stores useful results in long-term memory for future retrieval.

    args:
    - question: a question about game industry. 

    Returns web search results and saves top results to learned knowledge collection.
    """

    client = TavilyClient(api_key=TAVILY_API_KEY)

    # perform web search
    try:
        search_results = client.search(
            query=question,
            num_results=5
        )
    except Exception as e:
        return f"Web search failed: {str(e)}"

    # format results
    if not search_results.get('results'):
        return "No web results found."
    
    formatted_results = []
    for i, result in enumerate(search_results['results'], 1):
        formatted_results.append(
            f"Result {i}:\n"
            f" Title: {result.get('title', 'N/A')}\n"
            f" URL: {result.get('url', 'N/A')}\n"
            f" Content: {result.get('content', 'N/A')}\n"
        )
    
    # LONG-TERM MEMORY: Store top results in learned collection
    try:
        import time

        # Store top 2 results for future retrieval
        stored_count = 0

        for i, result in enumerate(search_results['results'][:2]):

            if result.get('content'): # only store if there's content
                # create document text
                learned_content = f"{result.get('title', 'Untitled')} - {result.get('content', '')}"

                # generate unique ID
                doc_id = f"learned_{int(time.time())}_{i}"

                # add to learned collection with metadata
                learned_collection.add(
                    ids = [doc_id],
                    documents = [learned_content],
                    metadatas = [{
                        "source": "web_search",
                        "url": result.get('url', ''),
                        "title": result.get('title', 'Untitled'),
                        "query": question,
                        "timestamp": time.time(),
                        "relevance_score": result.get('score', 0)
                    }]
                )
                stored_count += 1

        if stored_count > 0:
            formatted_results.append(
                f"\nLONG-TERM MEMORY: Stored {stored_count} result(s) for future retrieval."
            )

    except Exception as e:

        # Log any details if storage fails, but don't interrupt the main function
        formatted_results.append(
            f"\nNOTE: Could not store in long-term memory: {str(e)}"
        )

    return "\n".join(formatted_results)
    

In [58]:
# Test the web search tool directly
result = game_web_search("latest Nintendo game 2025")
print(result) # expected output: formatted web results with URLs

Result 1:
 Title: Catch up on some of 2025's game releases - Nintendo
 URL: https://www.nintendo.com/au/news-and-articles/catch-up-on-some-of-2025s-game-releases/?srsltid=AfmBOoq13-IUyiiK8SZ_HeZequqk9971rd568Lm_05-l8bh8CAdyU5KV
 Content: Catch up on some of 2025's game releases · ENDER MAGNOLIA: Bloom in the Mist · Xenoblade Chronicles X: Definitive Edition · The Hundred Line -

Result 2:
 Title: The Top 20 Switch & Switch 2 Games Of 2025! - YouTube
 URL: https://www.youtube.com/watch?v=oJOla_wrFhY
 Content: Some great games released for the Nintendo Switch but with the release of the Nintendo ... Ranking EVERY 2025 Nintendo Game. Crazytendo•13K views.

Result 3:
 Title: TOP 25 BEST NEW Nintendo Switch Games of 2025 - YouTube
 URL: https://www.youtube.com/watch?v=SdBeOymuADQ
 Content: ... NEW Nintendo Switch Games of 2025 2025 has been an AMAZING year for Nintendo Switch! From classic fighting game collections to brand new

Result 4:
 Title: Nintendo Switch games coming in 2025 | News


### Agent

In [59]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
agent = Agent( # Agent class uses StateMachine internally (lib/agents.py)
    model_name="gpt-4o-mini",
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.7,
    instructions="""
        You are UdaPlay, an AI research agent specializing in video game industry knowledge with long-term memory.

        Your workflow is as follows:
        1. When asked a question, ALWAYS start by using 'retrieve_game' to search your internal knowledge base
            - This searches BOTH:
                a) Original game database (from initial dataset)
                b) Learned knowledge (from previous web searches you've performed)
        2. Use 'evaluate_retrieval' to assess if the retrieved documents sufficiently answer the question
        3. If evaluation indicates documents are useful (useful=True):
            - provide a comprehensive answer based on the retrieved information
            - cite your sources appropriately (see citation rules below)
        4. If evaluation indicates documents are NOT useful (useful=False):
            - use 'game_web_search' to find current information
            - this will AUTOMATICALLY store useful results in your long-term memory
            - future queries can then retrieve this learned knowledge
        5. Citation Rules (IMPORTANT):
           - For original database: "According to my internal game database..."
           - For learned knowledge: "Based on information I previously learned from [source]..."
           - For fresh web search: "According to [source name/URL]..."
        6. When new information is stored in long-term memory, briefly acknowledge it:
            - Example: "I've stored this information for future reference."
        7. Be conversational and helpful, but accurate, pithy and factual
        8. If you cannot find relevant information even after searching learned knowledge and web search, 
            clearly state the limitations

        UNIQUE_CAPABILITY: Your long-term memory allows you to learn and improve oger time!
        Information you learn from web searches is permanently stored and will be available in all
        future conversations, even across different sessions.

        REMEMBER: You have memory of conversation history, so you can reference previous exchanges.
        Maintain context across multiple questions in the same session.
    """    
)

print("\nAgent configured with long-term memory capabilities")


Agent configured with long-term memory capabilities


In [60]:
# Citation Validation Test
def validate_citations(run, query_description):
    """
    Validates that the agent's response includes proper citations, and detects
    memory-based responses vs. new retrieval
    
    args:
        run: The Run object from agent.invoke()
        query_description: Description of what the query is about
        
    Returns:
        dict with validation results
    """
    
    final_state = run.get_final_state()
    messages = final_state["messages"]

    # get the final answer
    final_answer = None

    for msg in reversed(messages):
        if hasattr(msg, 'content') and msg.content and msg.role == "assistant":
            final_answer = msg.content
            break
    
    if not final_answer:
        return {
            "has_citation": False,
            "citation_type": None,
            "reason": "No final answer found",
            "query": query_description,
            "from_memory": False
        }
    
    # check if any tools were called the current run
    tools_used = []
    for msg in messages:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                tools_used.append(tc.function.name)

    # detect memory-based responses (no tools called)
    from_memory = len(tools_used) == 0
        
    # check for citation patterns
    citation_patterns = {
        "database": [
            "according to my game database",
            "according to my database",
            "based on my game database",
            "from my internal database",
            "from the database"
        ],
        "web": [
            "according to",
            "source:",
            "url:",
            "https://",
            "http://"
            "based on web search",
            "according to the web results"
        ],
        "memory": [
            "as I mentioned earlier",
            "as mentioned previously",
            "I have already told you",
            "from our earlier conversation",
            "as prviously discussed"
        ]
    }

    # convert to lowercase for easier matching
    answer_lower = final_answer.lower()

    # check for database citations
    has_db_citation = any(pattern in answer_lower for pattern in citation_patterns["database"])

    # check for web citations
    has_web_citation = any(pattern in answer_lower for pattern in citation_patterns["web"])

    # check from memory
    has_memory_reference = any(pattern in answer_lower for pattern in citation_patterns["memory"])

    # determine citation type
    if from_memory:
        citation_type = "memory (no new retrieval)"
    elif has_db_citation and has_web_citation:
        citation_type = "both"
    elif has_db_citation:
        citation_type = "database"
    elif has_web_citation:
        citation_type = "web"
    elif has_memory_reference:
        citation_type = "memory reference"
    else:
        citation_type = None   

    has_citation = has_db_citation or has_web_citation or has_memory_reference

    # Build reason
    if from_memory:
        reason = "No tools called, response likely based on memory"
    elif has_citation:
        reason = "Citation found"
    else:
        reason = "No citation found"

    return {
        "has_citation": has_citation,
        "citation_type": citation_type,
        "tools_used": list(set(tools_used)), # unique tools used
        "reason": reason,
        "query": query_description,
        "from_memory": from_memory,
        "answer_preview": final_answer[:200] + "..." if len(final_answer) > 200 else final_answer
    }

In [61]:
# Helper function to display results
# Also incorporates citation validation from validate_citations function
def print_agent_output(run, query_num, question):
    """
    Display agent output and validate citations; also notifies memory-based responses
    """
    print(f"\n{'='*80}")
    print(f"QUERY {query_num}: {question}")
    print(f"{'='*80}")

    final_state = run.get_final_state()
    messages = final_state["messages"]

    # show tool usage
    print("\n--- Tool Usage ---")
    tool_count = 0
    for msg in messages:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  - Called: {tc.function.name}")
                tool_count += 1
    
    if tool_count == 0:
        print(" NOTE: No tools were called; response may be based on memory.")

    # show final answer
    print("\n--- Final Answer ---")
    final_answer = messages[-1].content if messages else "No repsonse generated."
    
    print(final_answer)
    print(f"{'='*80}\n")

    # validate citations in the final answer
    print("\n--- Citation Validation ---")
    validation = validate_citations(run, question)

    if validation["from_memory"]:
        print(f" NOTE: Response from conversation memory")
        print(f". Tools used: {validation['tools_used'] or 'None'}")
        print(f". Recommendation: reset session or use unique session_id")
    elif validation["has_citation"]:
        print(f"Status: Citation found ({validation['citation_type']})")
    else:
        print(f"Status: No citation detected")
        print(f"  Reason: {validation['reason']}")
        print(f"  Tools used: {validation['tools_used']}")

    print(f"{'='*80}\n")

    return validation

In [63]:
# TODO: Invoke your agent
print("\n" + "="*80)
print("TESTING AGENT ON VARIOUS QUERIES")
print("="*80)

# uncomment to reset session before testing same id again
#agent.reset_session("test_session")

validation_results = [] # to store citation validation results for each query

# Query Run 1 - Internal Knowledge
run1 = agent.invoke(
    query="When Pokémon Gold and Silver was released?",
    session_id="test_session"
)
val1 = print_agent_output(run1, 1, "When Pokémon Gold and Silver was released?")
validation_results.append(val1)

# Query Run 2 - Specific detailed knowledge
run2 = agent.invoke(
    query="Which one was the first 3D platformer Mario game?",
    session_id="test_session"
)
val2 = print_agent_output(run2, 2, "Which one was the first 3D platformer Mario game?")
validation_results.append(val2)

# Query Run 3 - Current information (requires web search)
run3 = agent.invoke(
    query="Was Mortal Kombat X realeased for Playstation 5?",
    session_id="test_session"
)
val3 = print_agent_output(run3, 3, "Was Mortal Kombat X realeased for Playstation 5?")
validation_results.append(val3)

# Test Conversation Memory
print("\n" + "="*80)
print("TEST AGENT ON CONVERSATION MEMORY")
print("="*80)

# uncomment to reset session before testing same id again
#agent.reset_session("memory_test_session")

# Initial question
print("\n--- Initial Question ---")
print("QUERY 1A: What platform was Goldeneye 007 released on?")
run_m1 = agent.invoke(
    query="What platform was Goldeneye 007 released on?",
    session_id="memory_test_session"
)
print(f"\nAnswer: {run_m1.get_final_state()['messages'][-1].content}\n")

# Follow-up question referencing previous answer
print("\n--- Follow-up Question ---")
print("QUERY 1B: And what is its Metacritic score?")
run_m2 = agent.invoke(
    query="And what is its Metacritic score?", # changed to a different question to test memory reference without being too obvious
    session_id="memory_test_session"
)
print(f"Answer: {run_m2.get_final_state()['messages'][-1].content}\n")

print("\n" + "="*80)
print("Conversation continuity confirmed - agent maintains context across queries in the same session.")


 


TESTING AGENT ON VARIOUS QUERIES
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

QUERY 1: When Pokémon Gold and Silver was released?

--- Tool Usage ---
  - Called: retrieve_game
  - Called: evaluate_retrieval

--- Final Answer ---
Pokémon Gold and Silver were released in 1999 for the Game Boy Color. These games are part of the second generation of Pokémon and introduced new regions, Pokémon, and gameplay mechanics.


--- Citation Validation ---
Status: No citation detected
  Reason: No citation found
  Tools used: ['evaluate_retrieval', 'retrieve_game']

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachin

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes

In [65]:
# Long-term Memory Testing (ADVANCED FEATURE)
print("\n" + "="*80)
print("Testing: Agent learns from web searches and improves future responses\n")

# use a different session_id for this test
ltm_session = "ltm_test"

# First query - will need web search (not in original 15 games)
print("\n--- First Query: Learn New Information ---")
run_ltm1 = agent.invoke(
    query="What is the Metacritic score for The Witcher 3?",
    session_id=ltm_session
)
print_agent_output(run_ltm1, 1, "What is the Metacritic score for The Witcher 3?") # first time learning

# Second query - should retrieve from learned knowledge
print("\n--- Second Query: Same Topic (Should Use Learned Knowledge) ---")
run_ltm2 = agent.invoke(
    query="Tell me about The Witcher 3's critical reception",
    session_id=ltm_session
)
print_agent_output(run_ltm2, 2, "Tell me about The Witcher 3's critical reception") # should show retrieval from learned knowledge

# check what was learned
print("\n--- Learned Knowledge Verification ---")
learned_docs = learned_collection.get()
print(f"Total learned documents: {len(learned_docs['ids'])}")
print(f"Recently learned topics: {[meta.get('title', 'N/A')[:60] for meta in learned_docs['metadatas'][-3:]]}")
print("\nLong-term memory feature successfully demonstrated!")


Testing: Agent learns from web searches and improves future responses


--- First Query: Learn New Information ---
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

QUERY 1: What is the Metacritic score for The Witcher 3?

--- Tool Usage ---
  - Called: retrieve_game
  - Called: game_web_search

--- Final Answer ---
The Metacritic score for **The Witcher 3: Wild Hunt** remains at **97** based on 79 critic reviews, indicating "universal acclaim." The user score is also highly positive. If you need more details, feel free to ask!


--- Citation Validation ---
Status: No citation detected
  Reason: No citation found
  Tools used: ['retrieve_game', 'game_web_search']


--- Second Query: Same Topic (Should Use Learned Knowledge) ---
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Exe

In [66]:
# View Learned Knowledge Database
def view_learned_knowledge():
    """Display all learned knowledge in the database"""

    learned = learned_collection.get()
    print(f"LEARNED KNOWLEDGE DATABASE ({len(learned['ids'])} entries)")
    print("-"*80)

    for i, (doc_id, doc, meta) in enumerate(zip(learned['ids'], learned['documents'], learned['metadatas'])):
        print(f"\n[{i+1}] ID: {doc_id}")
        print(f"Title: {meta.get('title', 'N/A')}")
        print(f"Source: {meta.get('source', 'N/A')}")
        print(f"Query: {meta.get('query', 'N/A')}")
        print(f"Timestamp: {meta.get('timestamp', 'N/A')}")
        print(f"Content Preview: {doc[:200]}...")
        print("-"*80)

# uncomment to view all learned knowledge
view_learned_knowledge()

LEARNED KNOWLEDGE DATABASE (12 entries)
--------------------------------------------------------------------------------

[1] ID: learned_1772661216_0
Title: Catch up on some of 2025's game releases - Nintendo
Source: web_search
Query: latest Nintendo game 2025
Timestamp: 1772661216.722811
Content Preview: Catch up on some of 2025's game releases - Nintendo - Catch up on some of 2025's game releases · ENDER MAGNOLIA: Bloom in the Mist · Xenoblade Chronicles X: Definitive Edition · The Hundred Line -...
--------------------------------------------------------------------------------

[2] ID: learned_1772661217_1
Title: The Top 20 Switch & Switch 2 Games Of 2025! - YouTube
Source: web_search
Query: latest Nintendo game 2025
Timestamp: 1772661217.560779
Content Preview: The Top 20 Switch & Switch 2 Games Of 2025! - YouTube - Some great games released for the Nintendo Switch but with the release of the Nintendo ... Ranking EVERY 2025 Nintendo Game. Crazytendo•13K view...
------------------